# Mi Spotify Wrapped — EDA Personal
**Universidad de Pamplona · Bases de Datos II · 2026-I**
**Autor: Jherson Sanchez**

Análisis exploratorio sobre datos reales extraídos de la cuenta de Spotify
de Jherson Sanchez, almacenados en el Data Warehouse (PostgreSQL Neon).

Pipeline: `Spotify Web API → FastAPI ETL → dim_artists / dim_tracks / fact_listening_history`


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os
import warnings
warnings.filterwarnings("ignore")

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

SPOTIFY_GREEN = "#1DB954"
DARK_BG       = "#121212"
CARD_BG       = "#1E1E1E"
TEXT_COLOR     = "#FFFFFF"

plt.rcParams.update({
    "figure.facecolor":  DARK_BG,
    "axes.facecolor":    CARD_BG,
    "axes.edgecolor":    "#333333",
    "axes.labelcolor":   TEXT_COLOR,
    "xtick.color":       TEXT_COLOR,
    "ytick.color":       TEXT_COLOR,
    "text.color":        TEXT_COLOR,
    "grid.color":        "#2a2a2a",
    "grid.linestyle":    "--",
    "font.family":       "DejaVu Sans",
})

print("Conexion lista")


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Top artistas
df_artists = pd.read_sql(text("""
    SELECT
        a.name,
        a.popularity,
        a.followers_count,
        a.genres,
        COUNT(f.id) AS play_count
    FROM dwh.dim_artists a
    JOIN dwh.fact_listening_history f ON f.artist_id = a.artist_id
    JOIN dwh.dim_users u ON u.user_id = f.user_id
    GROUP BY a.artist_id, a.name, a.popularity, a.followers_count, a.genres
    ORDER BY play_count DESC
    LIMIT 20
"""), engine)

# Actividad por hora
df_hours = pd.read_sql(text("""
    SELECT hour_of_day, COUNT(*) AS plays
    FROM dwh.fact_listening_history
    GROUP BY hour_of_day
    ORDER BY hour_of_day
"""), engine)

# Top tracks
df_tracks = pd.read_sql(text("""
    SELECT
        t.name,
        a.name AS artist,
        t.popularity,
        t.duration_ms,
        COUNT(f.id) AS play_count
    FROM dwh.dim_tracks t
    JOIN dwh.dim_artists a ON a.artist_id = t.artist_id
    JOIN dwh.fact_listening_history f ON f.track_id = t.track_id
    GROUP BY t.track_id, t.name, a.name, t.popularity, t.duration_ms
    ORDER BY play_count DESC
    LIMIT 20
"""), engine)

# Actividad por dia
DAY_MAP = {"Monday":0,"Tuesday":1,"Wednesday":2,"Thursday":3,
           "Friday":4,"Saturday":5,"Sunday":6}
DAY_LABELS = ["Lun","Mar","Mie","Jue","Vie","Sab","Dom"]

df_days = pd.read_sql(text("""
    SELECT day_of_week, COUNT(*) AS plays
    FROM dwh.fact_listening_history
    GROUP BY day_of_week
"""), engine)
df_days["day_num"] = df_days["day_of_week"].map(DAY_MAP)
df_days = df_days.sort_values("day_num")

# Generos
df_genres = pd.read_sql(text("""
    SELECT UNNEST(a.genres) AS genre, COUNT(DISTINCT a.artist_id) AS artist_count
    FROM dwh.dim_artists a
    JOIN dwh.fact_listening_history f ON f.artist_id = a.artist_id
    JOIN dwh.dim_users u ON u.user_id = f.user_id
    WHERE a.genres IS NOT NULL AND array_length(a.genres,1) > 0
    GROUP BY genre
    ORDER BY artist_count DESC
    LIMIT 15
"""), engine)

print(f"Artistas cargados:      {len(df_artists)}")
print(f"Tracks cargados:        {len(df_tracks)}")
print(f"Horas con actividad:    {len(df_hours)}")
print(f"Generos unicos (top15): {len(df_genres)}")


## Vista general del DWH

In [ ]:
with engine.connect() as conn:
    total_plays   = conn.execute(text("SELECT COUNT(*) FROM dwh.fact_listening_history")).scalar()
    total_artists = conn.execute(text("SELECT COUNT(*) FROM dwh.dim_artists")).scalar()
    total_tracks  = conn.execute(text("SELECT COUNT(*) FROM dwh.dim_tracks")).scalar()
    peak_hour_row = conn.execute(text("""
        SELECT hour_of_day, COUNT(*) AS plays
        FROM dwh.fact_listening_history
        GROUP BY hour_of_day ORDER BY plays DESC LIMIT 1
    """)).fetchone()

peak_hour  = peak_hour_row[0]
peak_plays = peak_hour_row[1]

print("=" * 45)
print(f"  {'ESTADISTICAS DEL DWH':^43}")
print("=" * 45)
print(f"  Reproducciones totales:   {total_plays:>8,}")
print(f"  Artistas unicos:          {total_artists:>8,}")
print(f"  Canciones unicas:         {total_tracks:>8,}")
print(f"  Hora pico:                {peak_hour:>7}:00 hs  ({peak_plays} plays)")
print(f"  Avg duracion (tracks):    {df_tracks['duration_ms'].mean()/1000/60:>7.2f} min")
print(f"  Avg popularidad (tracks): {df_tracks['popularity'].mean():>7.1f} / 100")
print("=" * 45)


## Grafico 1 — Actividad por hora del dia
### El grafico que mas me sorprendio

La hipotesis inicial era escuchar musica en la manana o en la tarde temprana.
Los datos muestran algo diferente: el pico ocurre entre las 19:00 y las 20:00 hs,
con 57 y 58 reproducciones respectivamente. Esto revela un patron de escucha
nocturno muy marcado, concentrado despues de las 18:00.


In [ ]:
all_hours = pd.DataFrame({"hour_of_day": range(24)})
df_hours_full = all_hours.merge(df_hours, on="hour_of_day", how="left").fillna(0)
df_hours_full["plays"] = df_hours_full["plays"].astype(int)

peak_h = df_hours_full.loc[df_hours_full["plays"].idxmax(), "hour_of_day"]
max_plays = df_hours_full["plays"].max()

fig, ax = plt.subplots(figsize=(14, 5))

colors = [SPOTIFY_GREEN if h == peak_h else "#444444" for h in df_hours_full["hour_of_day"]]
ax.bar(df_hours_full["hour_of_day"], df_hours_full["plays"], color=colors, width=0.7, zorder=3)

ax.annotate(
    f"Pico: {int(peak_h):02d}:00 hs\n{int(max_plays)} plays",
    xy=(peak_h, max_plays),
    xytext=(peak_h - 4, max_plays - 8),
    fontsize=10, color=SPOTIFY_GREEN, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=SPOTIFY_GREEN),
)

ax.set_xlabel("Hora del dia", fontsize=11)
ax.set_ylabel("Reproducciones", fontsize=11)
ax.set_title("A que hora escucho mas musica — Jherson Sanchez", fontsize=14, fontweight="bold", pad=15)
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45, ha="right", fontsize=8)
ax.grid(axis="y", zorder=0)
ax.set_xlim(-0.5, 23.5)

fig.tight_layout()
plt.savefig("docs/assets/jherson_hora_pico.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()

print(f"Hora pico: {int(peak_h):02d}:00 hs con {int(max_plays)} reproducciones.")
print(f"El bloque 18:00-23:00 concentra el grueso de la actividad.")
print(f"Las horas de manana tienen muy poca o ninguna actividad.")


## Grafico 2 — Artistas mas escuchados

In [ ]:
top10 = df_artists.head(10).sort_values("play_count")

fig, ax = plt.subplots(figsize=(12, 6))

colors_a = [SPOTIFY_GREEN if v == top10["play_count"].max() else "#3a8a4a"
            if v >= top10["play_count"].quantile(0.75) else "#444444"
            for v in top10["play_count"]]

bars = ax.barh(top10["name"], top10["play_count"], color=colors_a, height=0.6)

for bar, val in zip(bars, top10["play_count"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(int(val)), va="center", fontsize=10, color=TEXT_COLOR, fontweight="bold")

ax.set_xlabel("Reproducciones en el DWH", fontsize=11)
ax.set_title("Top 10 artistas mas escuchados — Jherson Sanchez", fontsize=14, fontweight="bold", pad=15)
ax.set_xlim(0, top10["play_count"].max() * 1.25)
ax.grid(axis="x", zorder=0)

fig.tight_layout()
plt.savefig("docs/assets/jherson_top_artistas.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()

print(f"Arcangel lidera con {df_artists.iloc[0]['play_count']} reproducciones,")
print(f"muy por encima del segundo lugar ({df_artists.iloc[1]['play_count']} plays).")
print(f"Hay un artista claramente dominante en el historial.")


## Grafico 3 — Canciones mas reproducidas

In [ ]:
top_t = df_tracks.head(10).copy()
top_t["label"] = top_t["name"] + "\n(" + top_t["artist"] + ")"
top_t = top_t.sort_values("play_count")

fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.barh(top_t["label"], top_t["play_count"], color=SPOTIFY_GREEN, height=0.6)

for bar, val in zip(bars, top_t["play_count"]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            str(int(val)), va="center", fontsize=10, color=TEXT_COLOR, fontweight="bold")

ax.set_xlabel("Reproducciones", fontsize=11)
ax.set_title("Top 10 canciones mas reproducidas — Jherson Sanchez", fontsize=14, fontweight="bold", pad=15)
ax.set_xlim(0, top_t["play_count"].max() * 1.3)
ax.grid(axis="x", zorder=0)

fig.tight_layout()
plt.savefig("docs/assets/jherson_top_tracks.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()


## Grafico 4 — Generos dominantes

In [ ]:
top_g = df_genres.head(10)
palette = sns.color_palette("Greens_r", len(top_g))

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top_g["genre"][::-1], top_g["artist_count"][::-1],
               color=palette[::-1], height=0.6)

for bar, val in zip(bars, top_g["artist_count"][::-1]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            str(int(val)), va="center", fontsize=10, color=TEXT_COLOR)

ax.set_xlabel("Numero de artistas", fontsize=11)
ax.set_title("Generos mas presentes en el historial — Jherson Sanchez", fontsize=14, fontweight="bold", pad=15)
ax.grid(axis="x", zorder=0)

fig.tight_layout()
plt.savefig("docs/assets/jherson_generos.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()


## Grafico 5 — Popularidad vs reproducciones personales
### Que aprendi de mis propios datos que no sabia antes


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

scatter = ax.scatter(
    df_artists["popularity"],
    df_artists["play_count"],
    s=df_artists["followers_count"] / df_artists["followers_count"].max() * 800 + 60,
    c=df_artists["play_count"],
    cmap="Greens",
    alpha=0.85,
    edgecolors="#ffffff",
    linewidths=0.5,
    zorder=3,
)

for _, row in df_artists.head(10).iterrows():
    ax.annotate(
        row["name"],
        (row["popularity"], row["play_count"]),
        textcoords="offset points",
        xytext=(8, 3),
        fontsize=8,
        color=TEXT_COLOR,
        alpha=0.9,
    )

ax.axvline(x=50, color="#555555", linestyle="--", linewidth=1, label="Umbral popularidad media (50)")
ax.set_xlabel("Popularidad en Spotify (0-100)", fontsize=11)
ax.set_ylabel("Reproducciones en el historial", fontsize=11)
ax.set_title("Popularidad vs reproducciones personales\n(tamano = followers del artista)", fontsize=13, fontweight="bold", pad=15)
ax.legend(fontsize=9)
ax.grid(zorder=0)

plt.colorbar(scatter, ax=ax, label="Play count")
fig.tight_layout()
plt.savefig("docs/assets/jherson_popularidad_vs_plays.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()

low_pop  = df_artists[df_artists["popularity"] < 10]
high_pop = df_artists[df_artists["popularity"] >= 50]
print(f"Artistas con popularidad < 10:  {len(low_pop)} ({len(low_pop)/len(df_artists)*100:.0f}%)")
print(f"Artistas con popularidad >= 50: {len(high_pop)} ({len(high_pop)/len(df_artists)*100:.0f}%)")
print(f"\nDescubrimiento: Bad Bunny tiene la mayor popularidad (26) y")
print(f"Arcangel es el mas escuchado (60 plays) con solo popularidad 4.")
print(f"Esto muestra que el artista favorito personal no coincide con")
print(f"el mas popular a nivel global de Spotify.")


## Consultas analiticas del parcial

In [ ]:
print("=" * 50)
print("Pregunta 1 — En que hora del dia escuchas mas?")
print("=" * 50)
q1 = pd.read_sql(text("""
    SELECT hour_of_day, COUNT(*) AS reproducciones
    FROM dwh.fact_listening_history
    GROUP BY hour_of_day
    ORDER BY reproducciones DESC
    LIMIT 5
"""), engine)
print(q1.to_string(index=False))

print("\n" + "=" * 50)
print("Pregunta 2 — Cual es tu artista mas escuchado?")
print("=" * 50)
q2 = pd.read_sql(text("""
    SELECT a.name, COUNT(*) AS veces
    FROM dwh.fact_listening_history f
    JOIN dwh.dim_artists a ON a.artist_id = f.artist_id
    GROUP BY a.name
    ORDER BY veces DESC
    LIMIT 5
"""), engine)
print(q2.to_string(index=False))

print("\n" + "=" * 50)
print("Pregunta 3 — Cual es tu cancion mas repetida?")
print("=" * 50)
q3 = pd.read_sql(text("""
    SELECT t.name, a.name AS artista, COUNT(*) AS veces
    FROM dwh.fact_listening_history f
    JOIN dwh.dim_tracks t ON t.track_id = f.track_id
    JOIN dwh.dim_artists a ON a.artist_id = f.artist_id
    GROUP BY t.name, a.name
    ORDER BY veces DESC
    LIMIT 5
"""), engine)
print(q3.to_string(index=False))

print("\n" + "=" * 50)
print("Pregunta 4 — Que genero domina tu historial?")
print("=" * 50)
q4 = pd.read_sql(text("""
    SELECT UNNEST(a.genres) AS genero, COUNT(*) AS apariciones
    FROM dwh.dim_artists a
    JOIN dwh.fact_listening_history f ON f.artist_id = a.artist_id
    JOIN dwh.dim_users u ON u.user_id = f.user_id
    WHERE array_length(a.genres,1) > 0
    GROUP BY genero
    ORDER BY apariciones DESC
    LIMIT 5
"""), engine)
print(q4.to_string(index=False))

print("\n" + "=" * 50)
print("Pregunta 5 — En que dia escuchas mas musica?")
print("=" * 50)
q5 = pd.read_sql(text("""
    SELECT day_of_week, COUNT(*) AS reproducciones
    FROM dwh.fact_listening_history
    GROUP BY day_of_week
    ORDER BY reproducciones DESC
    LIMIT 5
"""), engine)
print(q5.to_string(index=False))


## Reflexiones finales

### 1. El grafico que mas me sorprendio

El grafico de actividad por hora del dia fue el resultado mas inesperado.
Se esperaba un perfil de escucha distribuido durante el dia, pero los datos
muestran una concentracion muy clara entre las 18:00 y las 23:00 hs,
con el pico maximo a las 20:00 con 58 reproducciones.
Las horas de manana (antes de las 14:00) tienen actividad casi nula.
Esto revela un patron de escucha exclusivamente nocturno.

---

### 2. Que aprendi de mis propios datos que no sabia antes

No sabia que mi consumo musical esta tan dominado por un solo artista.
Arcangel acumula 60 reproducciones, mas del doble que el segundo artista
(Eladio Carrion con 15). Ademas, su cancion "Quimica Sustancia" tiene
20 reproducciones — el track mas repetido por un margen amplio.

Tambien es notable la discrepancia entre popularidad global en Spotify
y popularidad personal: Bad Bunny (el artista con mayor indice de Spotify
en mi lista, con 26) aparece en el puesto 5 con solo 7 plays,
mientras que Arcangel con popularidad 4 encabeza el historial con 60 plays.
Esto muestra que mis preferencias no siguen el mainstream global.

---

### 3. Que pregunta quise hacerle a los datos y no pude

**Cual es la evolucion de mi escucha de Arcangel a lo largo del tiempo?
Siempre fue mi artista favorito, o es una fase reciente?**

El modelo actual no puede responder esto porque `fact_listening_history`
solo contiene las ultimas 50 reproducciones por la limitacion de
`GET /v1/me/player/recently-played`. No hay datos de meses anteriores.

Para responder esta pregunta se necesitaria agregar al modelo:

```sql
-- Nueva dimension temporal
CREATE TABLE dwh.dim_time (
    time_id   SERIAL PRIMARY KEY,
    date      DATE,
    week      INT,
    month     INT,
    year      INT,
    quarter   INT,
    day_type  VARCHAR(10)  -- 'weekday' / 'weekend'
);

-- Nueva columna en fact_listening_history
ALTER TABLE dwh.fact_listening_history ADD COLUMN time_id INT REFERENCES dwh.dim_time(time_id);
```

Adicionalmente se requeriria un sistema de scrobbling continuo que
registre cada reproduccion en tiempo real, no solo los ultimos 50 items
del historial de Spotify.
